# Quantum Machine Learning Model for Conspicuity Detection in Production: Data Preparation

We use a real-world industrial dataset (TIG aluminium 5083 weld images from Kaggle) to prepare data for a QML-based conspicuity (defect) detection model. This notebook only prepares the data: we download the dataset, select a labelled subset, and build a small project data folder. The actual QML model and training are implemented in Task 5.2.

## Environment Preparation

We need to install PennyLane and KaggleHub to our notebook first.

In [1]:
# %pip install pennylane # uncommment this to install
# %pip install kagglehub # uncommment this to install

We install and import the libraries we need. We use KaggleHub to download the dataset, and standard Python modules (os, json, shutil, etc.) to inspect paths, read labels, and copy files into our project data folder.

In [2]:
import pennylane as qml
from pennylane import numpy as np

import kagglehub
import os
import json
from collections import defaultdict
import shutil

## Downloading the Dataset

We download the TIG aluminium 5083 dataset from Kaggle using KaggleHub. This gives us a local path to the full dataset.

In [3]:
# download latest version
download_path = kagglehub.dataset_download("danielbacioiu/tig-aluminium-5083")
print("Path to downloaded dataset files:", download_path)

Path to downloaded dataset files: C:\Users\ASUS\.cache\kagglehub\datasets\danielbacioiu\tig-aluminium-5083\versions\1


## Inspecting the Dataset Structure

We inspect the downloaded directory layout to find where the training images and labels live.

In [4]:
# inspect downloaded files
print("Directories:", os.listdir(download_path))

Directories: ['al5083']


In [5]:
# inspect further
base_dir = os.path.join(download_path, 'al5083')
print("Directories:", os.listdir(base_dir))

Directories: ['README', 'test', 'train']


In [6]:
# get main directories
train_dir = os.path.join(base_dir, 'train')
test_dir  = os.path.join(base_dir, 'test')
print("Train path:", train_dir)
print("Test path:", test_dir)

Train path: C:\Users\ASUS\.cache\kagglehub\datasets\danielbacioiu\tig-aluminium-5083\versions\1\al5083\train
Test path: C:\Users\ASUS\.cache\kagglehub\datasets\danielbacioiu\tig-aluminium-5083\versions\1\al5083\test


## Loading Labels

We load the label file `train.json` from the train directory so that the relative image paths stored in it resolve correctly. This file maps each image path (relative to the train directory) to an integer label 0–5. The six classes are (as documented in the dataset README):

- 0: good weld
- 1: burn through
- 2: contamination
- 3: lack of fusion
- 4: misalignment
- 5: lack of penetration

In [7]:
dataset_root = train_dir
train_json_path = os.path.join(dataset_root, "train.json")

with open(train_json_path, "r") as f:
    train_meta = json.load(f)

print("Number of entries in train.json:", len(train_meta))

Number of entries in train.json: 26666


In [8]:
train_meta

{'170906-113317-Al 2mm-part3/frame_00647.png': 1,
 '170906-113317-Al 2mm-part3/frame_00672.png': 1,
 '170906-113317-Al 2mm-part3/frame_00677.png': 1,
 '170906-113317-Al 2mm-part3/frame_00646.png': 1,
 '170906-113317-Al 2mm-part3/frame_00691.png': 1,
 '170906-113317-Al 2mm-part3/frame_00684.png': 1,
 '170906-113317-Al 2mm-part3/frame_00665.png': 1,
 '170906-113317-Al 2mm-part3/frame_00668.png': 1,
 '170906-113317-Al 2mm-part3/frame_00651.png': 1,
 '170906-113317-Al 2mm-part3/frame_00655.png': 1,
 '170906-113317-Al 2mm-part3/frame_00657.png': 1,
 '170906-113317-Al 2mm-part3/frame_00687.png': 1,
 '170906-113317-Al 2mm-part3/frame_00654.png': 1,
 '170906-113317-Al 2mm-part3/frame_00676.png': 1,
 '170906-113317-Al 2mm-part3/frame_00688.png': 1,
 '170906-113317-Al 2mm-part3/frame_00674.png': 1,
 '170906-113317-Al 2mm-part3/frame_00649.png': 1,
 '170906-113317-Al 2mm-part3/frame_00693.png': 1,
 '170906-113317-Al 2mm-part3/frame_00643.png': 1,
 '170906-113317-Al 2mm-part3/frame_00662.png': 1,


We then build a list of (full_path, label) pairs so we can select images by class and later copy only the chosen subset into our project data folder.

In [9]:
# build full paths and labels
all_samples = []
for rel_path, label in train_meta.items():
    img_path = os.path.join(dataset_root, rel_path)
    all_samples.append((img_path, int(label)))  # ensure int label

In [10]:
# quick sanity check
print(all_samples[0])

('C:\\Users\\ASUS\\.cache\\kagglehub\\datasets\\danielbacioiu\\tig-aluminium-5083\\versions\\1\\al5083\\train\\170906-113317-Al 2mm-part3/frame_00647.png', 1)


## Choosing Samples

The full dataset is large and not all classes have the same number of images. To keep training time manageable and to balance classes, we cap the number of images per class. We group all samples by label, shuffle each group, take up to that many per class, then split each class into an 80% training and 20% test set. This way we get a smaller, class-balanced subset with a fixed train/test split for use in Task 5.2. We use a fixed random seed so the same subset is obtained on every run.

In [11]:
np.random.seed(42)

# group samples by label
by_class = defaultdict(list)
for img_path, label in all_samples:
    by_class[label].append(img_path)

for label in sorted(by_class):
    print(f"Label {label}: {len(by_class[label])} images available")

Label 0: 8758 images available
Label 1: 1783 images available
Label 2: 6325 images available
Label 3: 4028 images available
Label 4: 2953 images available
Label 5: 2819 images available


In [12]:
# select up to 300 images per class
max_per_class = 300
selected = {}

for label, paths in by_class.items():
    paths = np.array(paths)
    np.random.shuffle(paths)
    n = min(max_per_class, len(paths))
    selected[label] = paths[:n]

In [13]:
# train/test split per class
train_paths = []
train_labels = []
test_paths = []
test_labels = []

train_fraction = 0.8

for label, paths in selected.items():
    n = len(paths)
    n_train = int(train_fraction * n)
    train_paths.extend(paths[:n_train])
    train_labels.extend([label] * n_train)
    test_paths.extend(paths[n_train:])
    test_labels.extend([label] * (n - n_train))

print("Total train samples:", len(train_paths))
print("Total test samples:", len(test_paths))

Total train samples: 1440
Total test samples: 360


In [14]:
# ensure data types are Python lists of str and int
train_paths = [str(p) for p in train_paths]
train_labels = [int(l) for l in train_labels]
test_paths = [str(p) for p in test_paths]
test_labels = [int(l) for l in test_labels]

## Creating a Smaller Dataset

After creating a smaller training and test data sets, we create a local project data folder and copy only images of those data sets into it. We organize files by split and class so that Task 5.2 can load train and test images by class without touching the full 11GB download. This smaller folder is the data we use for the QML conspicuity detection notebook.

In [15]:
# create folder for data
subset_root = "conspicuity_production_data"
os.makedirs(subset_root, exist_ok=True)

# create subfolders
splits = {
    "train": (train_paths, train_labels),
    "test": (test_paths, test_labels),
}

for split_name, (paths, labels) in splits.items():
    for img_path, label in zip(paths, labels):
        # create a folder for each label
        class_dir = os.path.join(subset_root, split_name, str(label))
        os.makedirs(class_dir, exist_ok=True)

        # copy files into that folder
        filename = os.path.basename(img_path)
        dst_path = os.path.join(class_dir, filename)
        shutil.copy2(img_path, dst_path)

print("The subset of the dataset is copied to:", os.path.abspath(subset_root))

The subset of the dataset is copied to: C:\Users\ASUS\Desktop\Ongoing Projects\QML-for-Conspicuity-Detection-in-Production\conspicuity_production_data


# References

- Bacioiu, D., Melton, G., Papaelias, M., & Shaw, R. (2019). Automated defect classification of Aluminium 5083 TIG welding using HDR camera and neural networks. *Journal of Manufacturing Processes, 45*, 603–613. https://doi.org/10.1016/j.jmapro.2019.07.020

- Dataset link: [Kaggle](https://www.kaggle.com/datasets/danielbacioiu/tig-aluminium-5083?resource=download)

# Credits
This notebook is completed by `Myanmar Youths` for `Womanium Quantum + AI 2024` program.
- <a href="https://www.linkedin.com/in/lawun330/"><u>La Wun Nannda</u></a>
- <a href="https://www.linkedin.com/in/chit-zin-win-46a2a3263/"><u>Chit Zin Win</u></a>